In [4]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import  InMemorySaver
import time

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [3]:
class CrashState(TypedDict):
    input: str
    step1:str
    step2:str
    step3:str

In [5]:
def step1(state: CrashState) -> CrashState:
    print("Step 1: Processing input:", state['input'])
    return {'step1':'done','input':state['input']}

def step2(state: CrashState) -> CrashState:
    time.sleep(30)  
    print("Step 2: processing", state['input'])
    return {'step2':'done'}

def step3(state: CrashState) -> CrashState:
    print("Step 3: Finalizing", state['input'])
    return {'step3':'done'}

In [ ]:

graph = StateGraph(CrashState)


graph.add_node('step1',step1)
graph.add_node('step2',step2)
graph.add_node('step3',step3)

graph.add_edge(START,'step1')
graph.add_edge('step1','step2')
graph.add_edge('step2','step3')
graph.add_edge('step3',END)
checkpointer = InMemorySaver()
work = graph.compile(checkpointer=checkpointer)

In [15]:
try:
    print("Starting workflow execution...")
    workflow.invoke(None, config={"configurable": {"thread_id": "1"}})
except KeyboardInterrupt:
    print("Workflow execution interrupted. Saving state...")


Starting workflow execution...
Step 2: processing Initial input for processing
Step 3: Finalizing Initial input for processing


In [14]:
workflow.get_state(config={"configurable": {"thread_id": "1"}})

StateSnapshot(values={'input': 'Initial input for processing', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cef7a-2307-6238-8001-d813b73a6e68'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-23T15:37:19.852375+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cef7a-2304-6e2a-8000-aa7ab1c829e2'}}, tasks=(PregelTask(id='3d779246-109c-83da-2035-c104085828e5', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())